# T2.1 DBRepo schema creation notebook

This notebook creates the DBRepo database and empty normalized tables for the football-player market-value experiment. It uses the official DBRepo Python `RestClient`, which communicates with the DBRepo REST API. Fill in your DBRepo endpoint, username, password, and container id before running.


## Important

This notebook is for Role A / T2.1 only. It creates the schema. Loading the real data into the tables is mainly T2.5, owned by Role C.


In [40]:
# Install in your course environment if missing:
# %pip install dbrepo pandas

import os
import pandas as pd
from dbrepo.RestClient import RestClient


In [41]:
from dbrepo.RestClient import RestClient

client = RestClient(
    endpoint="https://test.dbrepo.tuwien.ac.at",
    username="12024699",
    password="Test123!"
)

containers = client.get_containers()

for c in containers:
    print(c)

id='6cfb3b8e-1792-4e46-871a-f3d103527203' name='mariadb-galera:11.3.2' image=ImageBrief(id='d79cb089-363c-488b-9717-649e44d8fcc5', name='mariadb', version='11.1.3', default=False) internal_name='mariadb_11_3_2' running=None hash=None


In [ ]:
# --- CONFIG: replace these values ---
DBREPO_ENDPOINT = os.getenv('DBREPO_ENDPOINT', 'https://test.dbrepo.tuwien.ac.at')
DBREPO_USERNAME = os.getenv('DBREPO_USERNAME', '12024699')
DBREPO_PASSWORD = os.getenv('DBREPO_PASSWORD', '#')
CONTAINER_ID = os.getenv('DBREPO_CONTAINER_ID', 'be8863ae-9794-4ee2-a964-3e22133b3840')

DATABASE_NAME = 'Predicting Football Player Market Value - Input Data'
IS_PUBLIC = True
IS_SCHEMA_PUBLIC = True


In [43]:
client = RestClient(DBREPO_ENDPOINT, username=DBREPO_USERNAME, password=DBREPO_PASSWORD)

database_id = "be8863ae-9794-4ee2-a964-3e22133b3840"

print("Using existing database id:", database_id)

Using existing database id: be8863ae-9794-4ee2-a964-3e22133b3840


## Metadata to use for the DBRepo record

The source datasets are external Mendeley Data records. Do not list TU Wien or the student group as original publisher/rightsholder of the input data. The group is only reusing and documenting the data for the course experiment.


In [44]:
source_dataset_metadata = [
    {
        'title': 'Forward football player valuation',
        'original_creators': 'Hugo Briseño; José Carlos Rivera',
        'publisher': 'Mendeley Data',
        'version': 'V1',
        'doi': '10.17632/cgc33scxg7.1',
        'doi_url': 'https://doi.org/10.17632/cgc33scxg7.1',
        'license': 'CC BY 4.0',
        'license_url': 'https://creativecommons.org/licenses/by/4.0/',
        'source_file': 'soccerplayers.xlsx'
    },
    {
        'title': 'Transfer Value Determinants',
        'original_creators': 'Ronald Nisanov',
        'publisher': 'Mendeley Data',
        'version': 'V2',
        'doi': '10.17632/3btg6ptc7b.2',
        'doi_url': 'https://doi.org/10.17632/3btg6ptc7b.2',
        'license': 'CC BY 4.0',
        'license_url': 'https://creativecommons.org/licenses/by/4.0/',
        'source_file': 'Nisanov_final_data.xlsx'
    }
]
source_dataset_metadata


[{'title': 'Forward football player valuation',
  'original_creators': 'Hugo Briseño; José Carlos Rivera',
  'publisher': 'Mendeley Data',
  'version': 'V1',
  'doi': '10.17632/cgc33scxg7.1',
  'doi_url': 'https://doi.org/10.17632/cgc33scxg7.1',
  'license': 'CC BY 4.0',
  'license_url': 'https://creativecommons.org/licenses/by/4.0/',
  'source_file': 'soccerplayers.xlsx'},
 {'title': 'Transfer Value Determinants',
  'original_creators': 'Ronald Nisanov',
  'publisher': 'Mendeley Data',
  'version': 'V2',
  'doi': '10.17632/3btg6ptc7b.2',
  'doi_url': 'https://doi.org/10.17632/3btg6ptc7b.2',
  'license': 'CC BY 4.0',
  'license_url': 'https://creativecommons.org/licenses/by/4.0/',
  'source_file': 'Nisanov_final_data.xlsx'}]

In [45]:
# DataFrame templates. DBRepo uses these to create table schemas.
# If your DBRepo instance rejects empty dataframes, add one temporary seed row and remove it after schema creation.

schemas = {
    'source_dataset': pd.DataFrame({
        'source_dataset_id': pd.Series(dtype='int64'),
        'dataset_title': pd.Series(dtype='string'),
        'original_creators': pd.Series(dtype='string'),
        'publisher': pd.Series(dtype='string'),
        'version_label': pd.Series(dtype='string'),
        'doi': pd.Series(dtype='string'),
        'doi_url': pd.Series(dtype='string'),
        'license_name': pd.Series(dtype='string'),
        'license_url': pd.Series(dtype='string'),
        'source_filename': pd.Series(dtype='string'),
        'notes': pd.Series(dtype='string'),
    }),
    'player': pd.DataFrame({
        'player_id': pd.Series(dtype='int64'),
        'player_name': pd.Series(dtype='string'),
    }),
    'club': pd.DataFrame({
        'club_id': pd.Series(dtype='int64'),
        'club_name': pd.Series(dtype='string'),
    }),
    'position': pd.DataFrame({
        'position_id': pd.Series(dtype='int64'),
        'position_name': pd.Series(dtype='string'),
    }),
    'nationality': pd.DataFrame({
        'nationality_id': pd.Series(dtype='int64'),
        'nationality_name': pd.Series(dtype='string'),
    }),
    'season': pd.DataFrame({
        'season_year': pd.Series(dtype='int64'),
        'notes': pd.Series(dtype='string'),
    }),
    'forward_player_valuation': pd.DataFrame({
        'forward_valuation_id': pd.Series(dtype='int64'),
        'source_dataset_id': pd.Series(dtype='int64'),
        'player_id': pd.Series(dtype='int64'),
        'club_id': pd.Series(dtype='int64'),
        'original_player_name': pd.Series(dtype='string'),
        'original_team_name': pd.Series(dtype='string'),
        'player_age_years': pd.Series(dtype='int64'),
        'market_value_mln': pd.Series(dtype='float64'),
        'value_rank': pd.Series(dtype='int64'),
        'plays_in_europe': pd.Series(dtype='bool'),
        'matches_played': pd.Series(dtype='int64'),
        'goals': pd.Series(dtype='int64'),
        'assists': pd.Series(dtype='int64'),
        'minutes_per_goal': pd.Series(dtype='int64'),
        'minutes_played': pd.Series(dtype='int64'),
        'instagram_followers_mln': pd.Series(dtype='float64'),
    }),
    'transfer_value_observation': pd.DataFrame({
        'transfer_observation_id': pd.Series(dtype='int64'),
        'source_dataset_id': pd.Series(dtype='int64'),
        'player_id': pd.Series(dtype='int64'),
        'position_id': pd.Series(dtype='int64'),
        'nationality_id': pd.Series(dtype='int64'),
        'club_id': pd.Series(dtype='int64'),
        'season_year': pd.Series(dtype='int64'),
        'original_player_name': pd.Series(dtype='string'),
        'original_position_name': pd.Series(dtype='string'),
        'original_nationality_name': pd.Series(dtype='string'),
        'original_club_name': pd.Series(dtype='string'),
        'age_then_years': pd.Series(dtype='int64'),
        'age_now_years': pd.Series(dtype='int64'),
        'club_performance': pd.Series(dtype='float64'),
        'relegation': pd.Series(dtype='bool'),
        'success_or_not': pd.Series(dtype='float64'),
        'total_games': pd.Series(dtype='int64'),
        'assists': pd.Series(dtype='int64'),
        'penalty_kicks': pd.Series(dtype='int64'),
        'total_minutes': pd.Series(dtype='int64'),
        'total_goals': pd.Series(dtype='int64'),
        'height_cm': pd.Series(dtype='float64'),
        'start_value_eur': pd.Series(dtype='float64'),
        'end_value_eur': pd.Series(dtype='float64'),
        'delta_value_eur': pd.Series(dtype='float64'),
        'value_start_mln': pd.Series(dtype='float64'),
        'value_end_mln': pd.Series(dtype='float64'),
        'value_delta_mln': pd.Series(dtype='float64'),
    }),
}


In [46]:
primary_keys = {
    "source_dataset": "source_dataset_id",
    "player": "player_id",
    "club": "club_id",
    "position": "position_id",
    "nationality": "nationality_id",
    "season": "season_year",
    "forward_player_valuation": "forward_valuation_id",
    "transfer_value_observation": "transfer_observation_id",
}

for table_name, primary_key in primary_keys.items():
    schemas[table_name] = schemas[table_name].set_index(primary_key)

print("Primary key indexes set.")

Primary key indexes set.


In [47]:
table_descriptions = {
    'source_dataset': 'Metadata about the reused external source datasets, including original creators, publisher, DOI, licence, source filename, and provenance notes.',
    'player': 'Unique football players appearing in the source datasets.',
    'club': 'Unique football clubs appearing in the source datasets.',
    'position': 'Unique football positions appearing in the Transfer Value Determinants dataset.',
    'nationality': 'Unique nationality labels appearing in the Transfer Value Determinants dataset.',
    'season': 'Season years represented in the Transfer Value Determinants workbook sheets.',
    'forward_player_valuation': 'Normalized observations from Forward football player valuation, linking players and clubs to age, value, match statistics, minutes, and Instagram information.',
    'transfer_value_observation': 'Normalized observations from Transfer Value Determinants, linking players, clubs, positions, nationalities, and seasons to performance and market-value variables.',
}


In [48]:
def make_dummy_dataframe(df):
    row = {}
    for col, dtype in df.dtypes.items():
        dtype_str = str(dtype)
        if "int" in dtype_str:
            row[col] = 0
        elif "float" in dtype_str:
            row[col] = 0.0
        elif "bool" in dtype_str:
            row[col] = False
        else:
            row[col] = "dummy"
    return pd.DataFrame([row])

In [49]:
primary_keys = {
    "source_dataset": "source_dataset_id",
    "player": "player_id",
    "club": "club_id",
    "position": "position_id",
    "nationality": "nationality_id",
    "season": "season_year",
    "forward_player_valuation": "forward_valuation_id",
    "transfer_value_observation": "transfer_observation_id",
}

def prepare_schema_df(table_name, df):
    pk = primary_keys[table_name]

    df = df.copy()

    if df.index.name != pk:
        df = df.set_index(pk)

    if df.empty:
        row = {}
        for col, dtype in df.dtypes.items():
            dtype_str = str(dtype)
            if "int" in dtype_str:
                row[col] = 0
            elif "float" in dtype_str:
                row[col] = 0.0
            elif "bool" in dtype_str:
                row[col] = False
            else:
                row[col] = "dummy"

        dummy_df = pd.DataFrame([row])
        dummy_df.index = pd.Index([1], name=pk)
        return dummy_df

    return df


created_tables = {}

for table_name, df in schemas.items():
    print("Creating table:", table_name)

    schema_df = prepare_schema_df(table_name, df)

    table = client.create_table(
        database_id,
        table_name,
        is_public=IS_PUBLIC,
        is_schema_public=IS_SCHEMA_PUBLIC,
        dataframe=schema_df,
        description=table_descriptions[table_name],
        with_data=False
    )

    created_tables[table_name] = table.id
    print("  table id:", table.id)

created_tables

Creating table: source_dataset
2026-05-14 15:30:43,033 root         WARNING default to 'text' for column dataset_title and type <class 'numpy.dtype'>
2026-05-14 15:30:43,033 root         WARNING default to 'text' for column original_creators and type <class 'numpy.dtype'>
2026-05-14 15:30:43,046 root         WARNING default to 'text' for column publisher and type <class 'numpy.dtype'>
2026-05-14 15:30:43,047 root         WARNING default to 'text' for column version_label and type <class 'numpy.dtype'>
2026-05-14 15:30:43,047 root         WARNING default to 'text' for column doi and type <class 'numpy.dtype'>
2026-05-14 15:30:43,054 root         WARNING default to 'text' for column doi_url and type <class 'numpy.dtype'>
2026-05-14 15:30:43,055 root         WARNING default to 'text' for column license_name and type <class 'numpy.dtype'>
2026-05-14 15:30:43,055 root         WARNING default to 'text' for column license_url and type <class 'numpy.dtype'>
2026-05-14 15:30:43,055 root        

{'source_dataset': '30ab5c29-fd9c-4e55-80cf-93b50c0aac8c',
 'player': '3e891984-2fef-407c-87d5-04e53fb064d0',
 'club': '5a3ea639-5db0-4b67-9043-8789290be13d',
 'position': '20f23de5-d87c-4fa1-b118-74b698a007a2',
 'nationality': 'dbfcb613-788b-49b5-bfd5-275c69441037',
 'season': '4a488515-49c0-493d-ae83-144736d9c27a',
 'forward_player_valuation': 'a1d877b3-9883-4e51-98fa-185ef004520a',
 'transfer_value_observation': 'a240d3ac-e3a2-43cb-b287-88d5fa43073f'}

## Optional: create a DBRepo identifier draft

Run this only after the database/table metadata looks correct in DBRepo. The exact DTO classes/enums available can depend on the DBRepo client version. If this cell needs small adjustments, use the DBRepo documentation for the installed version.


In [50]:
# Optional sketch; adapt to the exact installed DBRepo version if needed.
# from dbrepo.api.dto import (
#     IdentifierType, CreateIdentifierTitle, CreateIdentifierCreator,
#     CreateIdentifierDescription, License, Language, CreateRelatedIdentifier
# )
# identifier = client.create_identifier(
#     database_id=database_id,
#     type=IdentifierType.DATABASE,
#     titles=[CreateIdentifierTitle(title='Predicting Football Player Market Value - Input Data Schema')],
#     publisher='Mendeley Data / reused by Group 14 for FAIR Data Science course metadata',
#     creators=[
#         CreateIdentifierCreator(firstname='Hugo', lastname='Briseño'),
#         CreateIdentifierCreator(firstname='José Carlos', lastname='Rivera'),
#         CreateIdentifierCreator(firstname='Ronald', lastname='Nisanov'),
#         CreateIdentifierCreator(firstname='Konrad', lastname='Szegedy'),
#         CreateIdentifierCreator(firstname='Muhammad Athar', lastname='Riaz'),
#         CreateIdentifierCreator(firstname='Muhammad Bilal', lastname='Hussain'),
#         CreateIdentifierCreator(firstname='Ekene', lastname='Edeh'),
#     ],
#     publication_year=2026,
#     descriptions=[CreateIdentifierDescription(description='Normalized relational schema for reused football-player valuation datasets used in a course experiment.')],
# )
# print(identifier)
